<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI in Asset Management
## Chapter 7 · Data Engineering and Cleaning for Financial Time Series

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
Chapter 7 emphasizes robust data pipelines. Here we:
- inspect raw data quality,
- repair gaps and flag outliers,
- engineer reusable features, and
- persist cleaned panels for later notebooks.


### Getting Help While Engineering Data
- **Chapter 3** covers pandas fundamentals.
- **Appendix B** summarizes the rolling/resampling APIs used below.
- **Appendix F** contains practical tooling tips for organizing outputs.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8")
plt.rcParams.update({"font.family": "serif", "figure.dpi": 300})

DATA_PATH = Path("../data/pyaiam_eod.csv")
if not DATA_PATH.exists():
    DATA_PATH = "https://hilpisch.com/pyaiam_eod.csv"


## 1. Load Raw Prices and Profile Quality
We begin by checking for missing values and basic summary stats.

In [ ]:
prices = pd.read_csv(DATA_PATH, parse_dates=["Date"]).set_index("Date").sort_index()
summary = pd.DataFrame({
    "min": prices.min(),
    "max": prices.max(),
    "pct_missing": prices.isna().mean(),
})
summary

### 1.1 Gap Repair and Diagnostics
We forward-fill missing FX values and record which tickers required intervention.

In [ ]:
filled = prices.ffill()
fill_flags = prices.isna().cumsum().eq(1).any()
fill_flags

## 2. Outlier Detection
A rolling z-score flags extreme moves for manual review.

In [ ]:
log_rets = np.log(filled / filled.shift(1)).dropna()
zscores = (log_rets - log_rets.rolling(63).mean()) / log_rets.rolling(63).std()
outlier_mask = zscores.abs() > 4
outlier_counts = outlier_mask.sum().sort_values(ascending=False)
outlier_counts

### 2.1 Visualizing Flagged Points
We overlay outlier markers on a sample series.

In [ ]:
ticker = "BTC-USD"
fig, ax = plt.subplots(figsize=(12, 6))
filled[ticker].plot(ax=ax, alpha=0.7)
mask = outlier_mask[ticker].reindex(filled.index, fill_value=False)
ax.scatter(
    filled.index[mask],
    filled[ticker][mask],
    color="tomato",
    label="Outlier",
)
ax.set_title(f"Outlier Flags for {ticker}")
ax.legend()
plt.show()

## 3. Feature Engineering Helper
We compute rolling volatility, momentum, and z-scores for downstream models.

In [ ]:
def build_features(price_frame: pd.DataFrame, window: int = 21) -> pd.DataFrame:
    log_ret = np.log(price_frame / price_frame.shift(1))
    vol = log_ret.rolling(window).std() * np.sqrt(252)
    momentum = price_frame.pct_change(window)
    rolling_mean = price_frame.rolling(window).mean()
    rolling_std = price_frame.rolling(window).std()
    zscore = (price_frame - rolling_mean) / rolling_std
    out = pd.concat(
        {
            "vol": vol,
            "momentum": momentum,
            "zscore": zscore,
        },
        axis=1,
    )
    return out.dropna()

feature_panel = build_features(filled)
print(feature_panel.head().round(4))

### 3.1 Persist Clean Panels
Saving intermediate artifacts speeds up later notebooks.

In [ ]:
OUTPUT_DIR = Path("../data/features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
path = OUTPUT_DIR / "pyaiam_features.parquet"
feature_panel.to_parquet(path)
path

## 4. Data Quality Report Helper
Summaries like the one below fit nicely into Confluence or PDF appendices.

In [ ]:
def data_quality_report(raw: pd.DataFrame, cleaned: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "pct_missing_before": raw.isna().mean(),
            "pct_missing_after": cleaned.isna().mean(),
            "up_to_date": cleaned.iloc[-1].notna(),
        }
    )

dqr = data_quality_report(prices, filled)
dqr

## 5. Exercises
### Exercise 1 – Dynamic Thresholds
Modify the outlier detector to use a MAD-based threshold instead of z-scores.
<details><summary>Hint</summary>
Compute median absolute deviation via <code>rolling(window).apply</code>.
</details>

### Exercise 2 – Sector Aggregation
Assume a mapping from tickers to asset classes (reuse Chapter 1). Aggregate feature means by class each week.
<details><summary>Hint</summary>
Add a column with the mapping, then combine <code>groupby</code> with <code>resample</code>.
</details>

### Exercise 3 – File Versioning
Write a helper that appends a timestamped suffix to every feature file you save.
<details><summary>Hint</summary>
Use <code>datetime.now().strftime("%Y%m%d")</code> and create a new Path per run.
</details>


## 6. Takeaways for Chapter 7
- Basic QC checks catch missing data and outliers early.
- Encapsulating feature logic in functions makes reuse trivial.
- Persisting cleaned panels bridges the gap between research and production workflows.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">